# INTRODUCTION 

 ### Hotel Booking Cancellation Prediction - Prediction Demo

- This notebook demonstrates how the trained Gradient Boosting model can be used to predict whether a hotel booking is likely to be canceled.

The saved model and preprocessing transformer are loaded from the previous notebook and applied to new booking data.

In [1]:
import pandas as pd
import numpy as np
import joblib

In [4]:
best_model = joblib.load('../models/best_model.pkl')
transformer = joblib.load('../models/preprocessor.pkl')

print("Model and transformer loaded successfully.")

Model and transformer loaded successfully.


In [6]:
new_booking = pd.DataFrame([{
    'hotel': 'City Hotel',
    'lead_time': 120,
    'arrival_date_year': 2017,
    'arrival_date_month': 'August',
    'arrival_date_week_number': 32,
    'arrival_date_day_of_month': 10,
    'stays_in_weekend_nights': 1,
    'stays_in_week_nights': 3,
    'adults': 2,
    'children': 0,
    'babies': 0,
    'meal': 'BB',
    'country': 'PRT',
    'market_segment': 'Online TA',
    'distribution_channel': 'TA/TO',
    'is_repeated_guest': 0,
    'previous_cancellations': 0,
    'previous_bookings_not_canceled': 0,
    'reserved_room_type': 'A',
    'assigned_room_type': 'A',
    'booking_changes': 0,
    'deposit_type': 'No Deposit',
    'agent': 9.0,
    'days_in_waiting_list': 0,
    'customer_type': 'Transient',
    'adr': 100.0,
    'required_car_parking_spaces': 0,
    'total_of_special_requests': 1
}])

In [7]:
new_booking['total_guests'] = (
    new_booking['adults'] +
    new_booking['children'] +
    new_booking['babies']
)

new_booking['total_stay_nights'] = (
    new_booking['stays_in_weekend_nights'] +
    new_booking['stays_in_week_nights']
)

new_booking['is_weekend_stay'] = (
    new_booking['stays_in_weekend_nights'] > 0
).astype(int)

new_booking['has_previous_booking'] = (
    new_booking['previous_bookings_not_canceled'] > 0
).astype(int)

In [8]:
new_booking['arrival_date'] = pd.to_datetime(
    new_booking['arrival_date_year'].astype(str) + '-' +
    new_booking['arrival_date_month'] + '-' +
    new_booking['arrival_date_day_of_month'].astype(str)
)

In [10]:
print(new_booking.columns.tolist())

['hotel', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'total_stay_nights', 'is_weekend_stay', 'has_previous_booking', 'arrival_date']


In [14]:
# Feature Engineering

# 1. Total guests
new_booking['total_guests'] = (
    new_booking['adults'] +
    new_booking['children'] +
    new_booking['babies']
)

# 2. Total stay nights
new_booking['total_stay_nights'] = (
    new_booking['stays_in_weekend_nights'] +
    new_booking['stays_in_week_nights']
)

# 3. Total previous bookings
new_booking['total_previous_bookings'] = (
    new_booking['previous_cancellations'] +
    new_booking['previous_bookings_not_canceled']
)

# 4. Arrival date
new_booking['arrival_date'] = pd.to_datetime(
    new_booking['arrival_date_year'].astype(str) + '-' +
    new_booking['arrival_date_month'] + '-' +
    new_booking['arrival_date_day_of_month'].astype(str)
)

# 5. Day of week
new_booking['arrival_day_of_week'] = (
    new_booking['arrival_date'].dt.day_name()
)

# 6. Family indicator
new_booking['is_family'] = (
    (new_booking['children'] > 0) |
    (new_booking['babies'] > 0)
).astype(int)

# 7. Previous booking indicator
new_booking['has_previous_booking'] = (
    new_booking['total_previous_bookings'] > 0
).astype(int)

print("Feature engineering completed.")

Feature engineering completed.


In [15]:
feature_columns = transformer.feature_names_in_

new_booking = new_booking[feature_columns]

print("Input shape:", new_booking.shape)

Input shape: (1, 32)


In [21]:
new_booking = new_booking.copy()

new_booking['agent'] = new_booking['agent'].astype(str)

In [22]:
new_booking_transformed = transformer.transform(new_booking)

print("Transformed shape:", new_booking_transformed.shape)

Transformed shape: (1, 574)


In [17]:
for name, trans, columns in transformer.transformers_:
    print("\nTransformer:", name)
    print("Columns:", columns)


Transformer: num
Columns: Index(['lead_time', 'arrival_date_week_number', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'booking_changes',
       'days_in_waiting_list', 'adr', 'required_car_parking_spaces',
       'total_of_special_requests', 'total_guests', 'total_stay_nights',
       'total_previous_bookings', 'is_family', 'has_previous_booking'],
      dtype='object')

Transformer: cat
Columns: Index(['hotel', 'meal', 'country', 'market_segment', 'distribution_channel',
       'reserved_room_type', 'assigned_room_type', 'deposit_type', 'agent',
       'customer_type', 'arrival_day_of_week'],
      dtype='object')

Transformer: remainder
Columns: ['arrival_date']


In [23]:
prediction = best_model.predict(new_booking_transformed)[0]

probability = best_model.predict_proba(new_booking_transformed)[0, 1]

print("Prediction:", "Cancelled" if prediction == 1 else "Not Cancelled")
print(f"Cancellation Probability: {probability:.2%}")

Prediction: Not Cancelled
Cancellation Probability: 48.20%


- ### Prediction Observation

The model predicts that the given booking is **Not Cancelled**, with an estimated cancellation probability of **48.20%**.

Since the probability is below the classification threshold of 0.50, the model assigns the booking to the **Not Cancelled** class.

In [24]:
bookings = pd.concat([
    new_booking,
    new_booking.copy(),
    new_booking.copy()
], ignore_index=True)

In [25]:
bookings.loc[1, 'lead_time'] = 10
bookings.loc[1, 'total_of_special_requests'] = 3

bookings.loc[2, 'lead_time'] = 200
bookings.loc[2, 'deposit_type'] = 'Non Refund'

In [26]:
bookings_transformed = transformer.transform(bookings)

In [27]:
predictions = best_model.predict(bookings_transformed)
probabilities = best_model.predict_proba(bookings_transformed)[:, 1]

In [28]:
results = pd.DataFrame({
    'Booking': ['Booking 1', 'Booking 2', 'Booking 3'],
    'Cancellation Probability': probabilities,
    'Prediction': [
        'Cancelled' if p == 1 else 'Not Cancelled'
        for p in predictions
    ]
})

results['Cancellation Probability'] = (
    results['Cancellation Probability'] * 100
).round(2)

results

,Booking,Cancellation Probability,Prediction
0,Booking 1,48.20,Not Cancelled
1,Booking 2,27.66,Not Cancelled
2,Booking 3,85.91,Cancelled


### Prediction Demo Observation

The trained model successfully predicts the cancellation status of new hotel bookings using the same preprocessing pipeline applied during model training. The predicted probability provides an additional indication of the likelihood of cancellation.

## Conclusion

This notebook demonstrates the deployment-ready prediction workflow of the hotel booking cancellation model.

The trained model and preprocessing pipeline were loaded using Joblib. New booking data was transformed using the same feature engineering and preprocessing steps used during training, ensuring consistency between training and inference.

The model successfully generated both the predicted cancellation status and cancellation probability for new bookings.